# Phase 14: Cross-Dataset Schema Mapping

**Goal:** We have successfully explored 4 completely different datasets. But we have a massive problem: they all speak different languages! 
* NSL-KDD calls the attack label `label`.
* CICIDS-2017 calls it `Label`.
* UNSW-NB15 calls it `attack_cat`.
* BETH calls it `sus`.

Before we can build our Data Cleaning Pipeline (Phase 15) or train our AI, we must create a **Translation Layer** (A Unified Schema) so all 4 datasets speak the exact same language.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### Step 1: The Unified Target Schema (Subphase 14.1)
First, we need to design our "Universal Language". We will standardize on a specific set of feature names. 
If a dataset doesn't have one of these features, we will explicitly mark it as missing (`NaN`) so the AI knows how to handle it later.

In [2]:
# The official "Universal" feature names we will use for our entire AI project
UNIFIED_SCHEMA = [
    "unified_label",       # 0 for Normal, 1 for Attack
    "attack_category",     # E.g., 'DDoS', 'Worms', 'Normal'
    "protocol_type",       # TCP, UDP, ICMP
    "source_bytes",        # Bytes sent from the hacker
    "destination_bytes",   # Bytes received from the server
    "packet_count",        # Total packets sent
    "connection_duration", # How long the attack lasted
]

print("The Unified Schema has been established!")
print(f"All datasets will be forced into these {len(UNIFIED_SCHEMA)} universal columns.")

The Unified Schema has been established!
All datasets will be forced into these 7 universal columns.


### Step 2: Per-Dataset Mapping Dictionaries (Subphase 14.2)
Now we create the translation dictionaries. This tells pandas exactly how to rename the columns for each specific dataset so they perfectly match the Unified Schema.

In [3]:
# How to translate NSL-KDD into the Universal Language
NSLKDD_MAPPING = {
    "label": "attack_category",
    "protocol_type": "protocol_type",
    "src_bytes": "source_bytes",
    "dst_bytes": "destination_bytes",
    "duration": "connection_duration"
    # (Packet count is missing in NSL-KDD!)
}

# How to translate CICIDS-2017 into the Universal Language
CICIDS_MAPPING = {
    "Label": "attack_category",
    "Protocol": "protocol_type",
    "Total Length of Fwd Packets": "source_bytes",
    "Total Length of Bwd Packets": "destination_bytes",
    "Total Fwd Packets": "packet_count",  # Has packet count!
    "Flow Duration": "connection_duration"
}

# How to translate UNSW-NB15 into the Universal Language
UNSW_MAPPING = {
    "attack_cat": "attack_category",
    "proto": "protocol_type",
    "sbytes": "source_bytes",
    "dbytes": "destination_bytes",
    "spkts": "packet_count",
    "dur": "connection_duration"
}

# How to translate BETH into the Universal Language
BETH_MAPPING = {
    "sus": "unified_label", # BETH only has 0 or 1, no category name
    # BETH is a host dataset, so it is missing ALMOST ALL network features!
}

print("Translation Dictionaries Created Successfully!")

Translation Dictionaries Created Successfully!


### Step 3: Cross-Dataset Consolidated Comparison (Subphase 14.3)
To prove why this Unified Schema is necessary, let's create a side-by-side comparison table of all 4 datasets based on the EDA we just finished.

In [4]:
comparison_data = {
    "Dataset": ["NSL-KDD", "CICIDS-2017", "UNSW-NB15", "BETH"],
    "Total Rows": ["125,973", "2,830,743", "175,341", "763,144"],
    "Original Features": [41, 78, 42, 14],
    "Missing Target Features": ["packet_count", "None", "None", "All Network Features"],
    "Primary Imbalance": ["Moderate", "Extreme (Log Scale)", "Severe (Rare Attacks)", "Extreme (Drift Bursts)"],
    "Original Label Name": ["label", "Label", "attack_cat", "sus"]
}

comparison_df = pd.DataFrame(comparison_data)
print("=== CROSS-DATASET CONSOLIDATED COMPARISON TABLE ===\n")
display(comparison_df)

=== CROSS-DATASET CONSOLIDATED COMPARISON TABLE ===



,Dataset,Total Rows,Original Features,Missing Target Features,Primary Imbalance,Original Label Name
0,NSL-KDD,"125,973",41,packet_count,Moderate,label
1,CICIDS-2017,"2,830,743",78,None,Extreme (Log Scale),Label
2,UNSW-NB15,"175,341",42,None,Severe (Rare Attacks),attack_cat
3,BETH,"763,144",14,All Network Features,Extreme (Drift Bursts),sus


---
### Step 4: Schema Mapping Test Suite (Subphase 14.4)
Before we trust these dictionaries with millions of rows of real data, we need to test them. 
We will create a tiny piece of 'Fake' CICIDS-2017 data, push it through our translation layer, and write an `assert` test to mathematically prove the translation worked.

In [5]:
# 1. Create a fake CICIDS-2017 row of data
fake_cicids_data = pd.DataFrame([{
    "Label": "DDoS",
    "Protocol": "TCP",
    "Total Length of Fwd Packets": 500
}])

print("Original Fake Data (CICIDS Language):")
display(fake_cicids_data)

# 2. Apply our translation dictionary
mapped_data = fake_cicids_data.rename(columns=CICIDS_MAPPING)

print("\nAfter Schema Mapping (Universal Language):")
display(mapped_data)

# 3. The Test Suite (Asserts will crash the program if the mapping failed)
assert "attack_category" in mapped_data.columns, "Test Failed: attack_category missing!"
assert "protocol_type" in mapped_data.columns, "Test Failed: protocol_type missing!"
assert "source_bytes" in mapped_data.columns, "Test Failed: source_bytes missing!"

print("\n✅ SCHEMA MAPPING TEST SUITE PASSED SUCCESSFULLY!")

Original Fake Data (CICIDS Language):


,Label,Protocol,Total Length of Fwd Packets
0,DDoS,TCP,500



After Schema Mapping (Universal Language):


,attack_category,protocol_type,source_bytes
0,DDoS,TCP,500



✅ SCHEMA MAPPING TEST SUITE PASSED SUCCESSFULLY!


---
### Step 4: Conclusion & Handoff to the Cleaning Pipeline
We have successfully mapped out exactly how to unify these 4 radically different datasets. 

In **Phase 15 (Data Cleaning Pipeline)**, we will actually apply these translation dictionaries to the raw data, execute the renaming, handle all the `NaN` gaps, and prepare the final clean CSVs for the AI!